In [2]:
import shap
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import os
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

os.makedirs("../results/figures/shap", exist_ok=True)

C:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Use mBERT (your best performing model)
MODEL_PATH = "../models/transformers/mBERT_finetuned"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)
model.eval()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = model.to(DEVICE)
print(f"✅ Model loaded on {DEVICE}")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 3143.86it/s]


✅ Model loaded on cuda


In [9]:
def predict_proba(texts):
    """
    Handles both plain strings and
    SHAP masked numpy arrays
    """
    # ── Fix input type ────────────────────────────────────────
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()

    if isinstance(texts, str):
        texts = [texts]

    # Convert any non-string elements to string
    texts = [
        str(t) if not isinstance(t, str) else t
        for t in texts
    ]

    # Remove empty strings
    texts = [t if t.strip() else "[UNK]" for t in texts]

    all_probs = []
    batch_size = 8

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors = "pt",
            truncation     = True,
            padding        = True,
            max_length     = 128
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.softmax(
            outputs.logits, dim=1
        ).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)

# ── Test fixed function ───────────────────────────────────────
test_texts = [
    "Absolutely amazing product! Best purchase ever!!!",
    "khana ekdum mito thiyo, feri aaunchu pakkai",
]
probs = predict_proba(test_texts)
print("✅ predict_proba works!")
for text, prob in zip(test_texts, probs):
    label = "FAKE" if prob[1] > 0.5 else "GENUINE"
    print(f"  {label} ({prob[0]:.3f} genuine / "
          f"{prob[1]:.3f} fake)")
    print(f"  → {text[:60]}")

✅ predict_proba works!
  FAKE (0.000 genuine / 1.000 fake)
  → Absolutely amazing product! Best purchase ever!!!
  GENUINE (1.000 genuine / 0.000 fake)
  → khana ekdum mito thiyo, feri aaunchu pakkai


In [6]:
# Load real test data
real_test = pd.read_csv("../data/processed/real_test.csv")

# Load raw synthetic for language style
df_syn_raw = pd.read_csv("../data/processed/processed_reviews.csv")
syn_test   = pd.read_csv("../data/processed/syn_test.csv")

syn_test_lang = syn_test.merge(
    df_syn_raw[["review_text", "language_style"]],
    on="review_text", how="left"
)

# ── Pick diverse samples for SHAP analysis ────────────────────
shap_samples = []

# 2 genuine per language
for lang in ["english", "romanized_nepali", "code_mixed"]:
    subset = syn_test_lang[
        (syn_test_lang["label"] == 0) &
        (syn_test_lang["language_style"] == lang)
    ].head(2)
    shap_samples.append(subset)

# 2 fake per language
for lang in ["english", "romanized_nepali", "code_mixed"]:
    subset = syn_test_lang[
        (syn_test_lang["label"] == 1) &
        (syn_test_lang["language_style"] == lang)
    ].head(2)
    shap_samples.append(subset)

shap_df = pd.concat(shap_samples, ignore_index=True)
shap_texts  = shap_df["review_text"].tolist()
shap_labels = shap_df["label"].tolist()

print(f"✅ SHAP samples prepared : {len(shap_texts)}")
print(f"   Genuine : {shap_labels.count(0)}")
print(f"   Fake    : {shap_labels.count(1)}")
print(f"\nSample reviews:")
for i, (t, l) in enumerate(zip(shap_texts[:3], shap_labels[:3])):
    print(f"  [{i}] {'FAKE' if l==1 else 'GENUINE'}: {t[:60]}...")

✅ SHAP samples prepared : 12
   Genuine : 6
   Fake    : 6

Sample reviews:
  [0] GENUINE: i call repair guy and then he come latee latee and check slo...
  [1] GENUINE: thik thik. not wow. food edible. place ok. nothing more...
  [2] GENUINE: Keema noodles 601 set paisa vasool thyo cha Pachi pheri try ...


In [10]:
# Use pipeline-based explainer instead
# More stable with transformers

pipe = pipeline(
    "text-classification",
    model     = model,
    tokenizer = tokenizer,
    device    = 0 if torch.cuda.is_available() else -1,
    top_k     = None,        # return all class scores
)

def pipeline_predict(texts):
    """Pipeline-based predict for SHAP"""
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    if isinstance(texts, str):
        texts = [texts]
    texts = [
        str(t) if not isinstance(t, str) else t
        for t in texts
    ]
    texts = [t if t.strip() else "[UNK]" for t in texts]

    results = pipe(texts)
    probs   = []
    for result in results:
        # Sort by label to ensure consistent order
        sorted_result = sorted(result, key=lambda x: x["label"])
        probs.append([r["score"] for r in sorted_result])
    return np.array(probs)

# ── Recreate explainer with fixed function ────────────────────
masker    = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(
    pipeline_predict,
    masker,
    output_names=["LABEL_0", "LABEL_1"]  # genuine, fake
)

print("✅ Explainer recreated!")

# ── Test explainer on one sample ──────────────────────────────
test_val = pipeline_predict([
    "Absolutely amazing! Best product ever!!!",
    "ramro thiyo khana, feri aaunchu"
])
print(f"Test probabilities shape : {test_val.shape}")
print(f"Sample 1 : genuine={test_val[0][0]:.3f} "
      f"fake={test_val[0][1]:.3f}")
print(f"Sample 2 : genuine={test_val[1][0]:.3f} "
      f"fake={test_val[1][1]:.3f}")

✅ Explainer recreated!
Test probabilities shape : (2, 2)
Sample 1 : genuine=0.003 fake=0.997
Sample 2 : genuine=1.000 fake=0.000


In [8]:
# ⚠️ This takes a few minutes
# Start with small batch first

print("Computing SHAP values (this may take 5-15 min)...")
print("Using first 12 samples...")

# Compute SHAP values
shap_values = explainer(
    shap_texts[:12],
    max_evals = 500,    # higher = more accurate but slower
    batch_size = 4,
)

print(f"✅ SHAP values computed!")
print(f"   Shape : {shap_values.shape}")
# Shape: (n_samples, n_tokens, n_classes)

Computing SHAP values (this may take 5-15 min)...
Using first 12 samples...


ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).